# N64 PIMC Minimization Visualization

This notebook visualizes the minimization trajectory of a 64-particle Helium PIMC configuration.
Run `minimize_N64.py` first to generate the trajectory file.

## Initialization

In [ ]:
import sys
import os
import numpy as np
import pandas as pd

# Add parent directory to path to import from jax_landscape
sys.path.insert(0, os.path.abspath('../..'))

from jax_landscape.io.pimc import load_pimc_worldline_file

# Visualization libraries
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from ipywidgets import interact, IntSlider

## Load Trajectory and Log Data

In [ ]:
# System parameters (from test_full_wl)
N = 64
n = 0.0218  # Angstrom^-3
L = (N/n)**(1/3)  # Box size
box = [L, L, L]

# Load minimization trajectory
trajectory_file = 'N64_minimization_trajectory.dat'
log_file = 'N64_minimization.log'

print(f"Loading trajectory from {trajectory_file}...")
confs = load_pimc_worldline_file(trajectory_file, Lx=box[0], Ly=box[1], Lz=box[2])
print(f"✅ Loaded {len(confs)} configurations")

# Load log data
print(f"\nLoading energy log from {log_file}...")
log_data = pd.read_csv(log_file)
print(f"✅ Loaded {len(log_data)} log entries")
print(f"\nLog columns: {list(log_data.columns)}")
print(f"Iteration range: {log_data['Iteration'].min()} to {log_data['Iteration'].max()}")

# Get configuration indices (iteration numbers)
conf_indices = sorted(confs.keys())
print(f"\nConfiguration snapshots at iterations: {conf_indices[:5]}...{conf_indices[-3:]}")

## Energy Evolution Plot

In [ ]:
# Plot energy evolution
fig = make_subplots(
    rows=3, cols=1,
    subplot_titles=('Total Energy (Urp)', 'Spring Energy (E_sp)', 'Interaction Energy (E_int)'),
    vertical_spacing=0.1
)

# Total energy
fig.add_trace(
    go.Scatter(x=log_data['Iteration'], y=log_data['Energy(Urp)'], 
               mode='lines', name='Urp', line=dict(color='blue')),
    row=1, col=1
)

# Spring energy
fig.add_trace(
    go.Scatter(x=log_data['Iteration'], y=log_data['E_sp'], 
               mode='lines', name='E_sp', line=dict(color='green')),
    row=2, col=1
)

# Interaction energy
fig.add_trace(
    go.Scatter(x=log_data['Iteration'], y=log_data['E_int'], 
               mode='lines', name='E_int', line=dict(color='red')),
    row=3, col=1
)

# Add vertical lines for saved configurations
for iter_num in conf_indices:
    for row in [1, 2, 3]:
        fig.add_vline(x=iter_num, line_dash="dot", line_color="gray", opacity=0.3, row=row, col=1)

fig.update_xaxes(title_text="Iteration", row=3, col=1)
fig.update_yaxes(title_text="Energy (kB·K)", row=1, col=1)
fig.update_yaxes(title_text="Energy (kB·K)", row=2, col=1)
fig.update_yaxes(title_text="Energy (kB·K)", row=3, col=1)

fig.update_layout(height=800, showlegend=False, 
                  title_text="Energy Evolution During Minimization<br><sub>Vertical lines show saved trajectory snapshots</sub>")
fig.show()

# Print energy summary
print(f"\n📊 Energy Summary:")
print(f"Initial Urp: {log_data['Energy(Urp)'].iloc[0]:.2f} kB·K")
print(f"Final Urp: {log_data['Energy(Urp)'].iloc[-1]:.2f} kB·K")
print(f"Reduction: {log_data['Energy(Urp)'].iloc[0] - log_data['Energy(Urp)'].iloc[-1]:.2f} kB·K")
print(f"\nFinal E_sp: {log_data['E_sp'].iloc[-1]:.2f} kB·K")
print(f"Final E_int: {log_data['E_int'].iloc[-1]:.2f} kB·K")

## 3D Worldline Visualization

In [ ]:
def plot_worldline_3d(config_idx):
    """
    Plot the 3D coordinate space trace of worldline path.
    """
    conf_iter = conf_indices[config_idx]
    conf = confs[conf_iter]
    
    # Get energy at this iteration
    log_row = log_data[log_data['Iteration'] == conf_iter]
    if len(log_row) > 0:
        energy_urp = log_row['Energy(Urp)'].values[0]
        energy_sp = log_row['E_sp'].values[0]
        energy_int = log_row['E_int'].values[0]
    else:
        energy_urp = energy_sp = energy_int = np.nan
    
    print(f"🔍 Config at Iteration {conf_iter}")
    print(f"   M={conf.numTimeSlices}, N={conf.numParticles}")
    print(f"   Urp={energy_urp:.2f}, E_sp={energy_sp:.2f}, E_int={energy_int:.2f} kB·K")

    fig = go.Figure()

    # Get normalized cycle lengths
    cycle_lengths_normalized = conf.cycleSizeDist / conf.numTimeSlices
    unique_cycle_lengths = np.unique(cycle_lengths_normalized)

    # Create color mapping
    cycle_length_to_color_idx = {length: idx for idx, length in enumerate(unique_cycle_lengths)}
    num_unique_lengths = len(unique_cycle_lengths)

    # Group beads by cycle length
    beads_by_cycle_length = {length: {'x': [], 'y': [], 'z': []} for length in unique_cycle_lengths}

    for m in range(conf.numTimeSlices):
        for n in range(conf.numParticles):
            cycle_idx = conf.cycleIndex[m, n]
            cycle_length = cycle_lengths_normalized[cycle_idx]
            beads_by_cycle_length[cycle_length]['x'].append(conf.beadCoord[m, n, 0])
            beads_by_cycle_length[cycle_length]['y'].append(conf.beadCoord[m, n, 1])
            beads_by_cycle_length[cycle_length]['z'].append(conf.beadCoord[m, n, 2])

    # Plot each cycle length group
    for cycle_length in unique_cycle_lengths:
        color_idx = cycle_length_to_color_idx[cycle_length]
        color = f'hsl({(color_idx * 360 / num_unique_lengths) % 360}, 80, 50)'

        coords = beads_by_cycle_length[cycle_length]
        fig.add_trace(go.Scatter3d(
            x=coords['x'],
            y=coords['y'],
            z=coords['z'],
            mode='markers',
            marker=dict(size=2, color=color),
            name=f'cycle {cycle_length:.0f}'
        ))

    # Configure layout
    title_text = f"Minimization Iteration {conf_iter}<br><sub>Urp={energy_urp:.1f}, E_sp={energy_sp:.1f}, E_int={energy_int:.1f} kB·K</sub>"

    fig.update_layout(
        title=dict(text=title_text, x=0.5, font=dict(size=14)),
        scene=dict(
            xaxis_title="X (Å)", yaxis_title="Y (Å)", zaxis_title="Z (Å)",
            aspectmode='cube',
            bgcolor='rgba(240,240,240,0.1)',
            xaxis=dict(showgrid=True, gridcolor='lightgray', range=[0, box[0]]),
            yaxis=dict(showgrid=True, gridcolor='lightgray', range=[0, box[1]]),
            zaxis=dict(showgrid=True, gridcolor='lightgray', range=[0, box[2]])
        ),
        showlegend=True, legend=dict(x=0.02, y=0.98),
        width=900, height=700
    )

    fig.show()
    
    # Print cycle statistics
    print(f"   Cycles: {conf.cycleSizeDist.shape[0]}, unique lengths: {unique_cycle_lengths}")

print("✅ Visualization function ready")

## Interactive Slider

In [ ]:
print("🎯 Interactive Minimization Trajectory Visualization")
print(f"Use slider to view {len(conf_indices)} saved configurations")

w = interact(
    plot_worldline_3d, 
    config_idx=IntSlider(
        min=0, 
        max=len(conf_indices)-1, 
        step=1, 
        value=0,
        description='Snapshot:'
    )
)
display(w)